# 06 — EXP_10: Passage-level LIME (Leave-One-Out attribution)

**Method.** For each question, take the architecture's retrieved chunks, build the full prompt + `k` leave-one-out prompts, run them through Groq, and compute per-passage attribution from the score deltas.

**Score functions:**
- `correctness_attribution[i]` = score_full − score_loo_i (binary: 1 if pred matches gold, else 0).
- `sameletter_attribution[i]` = 1 if removing chunk `i` changed the predicted letter, else 0.

**Cost:** $0 (all Groq). Per question: 1 full call (cached from EXP_02/04/05) + k LOO calls (fresh). For Multi-Hop with up to k=15 chunks, that's ~16 calls per question.

**Architectures considered** (No-RAG excluded — it has no chunks to attribute):
- EXP_02 Naive Dense (5 chunks)
- EXP_04 Hybrid RRF (5 chunks)
- EXP_05 Multi-Hop (up to 15 chunks)
- (Optional: Sparse, but it has the worst Context Precision; lowest-value target.)

**Sampling.** 200 questions stratified by complexity bucket (~58 Simple, ~62 Moderate, ~80 Complex, seed=42) drawn from the test_1273 split. Same sample is reused by EXP_11 SHAP + EXP_12 agreement for direct per-question comparison.

**Stages.** Smoke (3 questions × Multi-Hop, ~3 min) → small (50 questions × Multi-Hop, ~30 min) → full (200 questions × N architectures, ~scope-dependent). User runs each gate.

## 1. Setup

In [1]:
import json
import logging
import os
import sys
from functools import partial
from pathlib import Path

os.environ["ANONYMIZED_TELEMETRY"] = "False"
logging.getLogger("chromadb").setLevel(logging.WARNING)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env")

import pandas as pd

from src.data.loaders import load_chunks, load_medqa_4opt
from src.generation.groq_client import groq_complete, DEFAULT_MODEL, DEFAULT_TEMPERATURE
from src.retrieval.base import Chunk
from src.xai.lime_passage import run_lime_passage_batch

print("GROQ_API_KEY present:", "\u2713" if os.environ.get("GROQ_API_KEY") else "\u2717")
print("Repo root:", REPO_ROOT)

GROQ_API_KEY present: ✓
Repo root: /Users/rajak/Workstation/Projects/myGitHub/thesis-project


## 2. Sample 200 stratified test questions (deterministic, reused across EXP_10/11/12)

Stratification proportions match the bucket distribution on test_1273 (29 / 31 / 40 %). Seed=42 so EXP_11 + EXP_12 use the exact same sample.

In [2]:
SAMPLE_N = 200
SAMPLE_SEED = 42

medqa = load_medqa_4opt()
medqa = medqa.reset_index(drop=False).rename(columns={"index": "row_idx"})
medqa["question_id"] = "medqa_" + medqa["row_idx"].astype(str)

labels = pd.read_parquet(REPO_ROOT / "data/processed/complexity_labels.parquet")
test = medqa[medqa.split == "test"].merge(
    labels[["question_id", "complexity"]].astype({"complexity": str}),
    on="question_id",
)

# Stratified sample: take ~SAMPLE_N proportional to bucket size
by_bucket = []
bucket_counts = test.complexity.value_counts().to_dict()
for b in ("Simple", "Moderate", "Complex"):
    pool = test[test.complexity == b]
    n = round(SAMPLE_N * len(pool) / len(test))
    if len(pool) >= n:
        by_bucket.append(pool.sample(n=n, random_state=SAMPLE_SEED))
SAMPLE = pd.concat(by_bucket).sample(frac=1, random_state=SAMPLE_SEED).reset_index(drop=True)

print(f"Sample size : {len(SAMPLE)}")
print(f"Bucket mix  : {SAMPLE.complexity.value_counts().to_dict()}")
print(f"Step mix    : {SAMPLE.meta_info.value_counts().to_dict()}")

Sample size : 201
Bucket mix  : {'Complex': 81, 'Moderate': 62, 'Simple': 58}
Step mix    : {'step1': 109, 'step2&3': 92}


## 3. Helpers — load per-architecture retrieved chunks for the sample

For each architecture, the `retrieval.jsonl` stores `(question_id, chunk_ids, chunk_scores)`. We join `chunk_ids` to `chunks.parquet` to get the chunk text and build `Chunk` objects.

In [3]:
chunks_df = load_chunks().set_index("chunk_id")
print(f"chunks.parquet : {len(chunks_df):,} chunks indexed")

def load_retrieval_for_sample(arch_prefix: str, sample_qids: list[str]) -> dict[str, list[Chunk]]:
    """Load retrieval.jsonl for an architecture, filter to the sample, return
    `{question_id: [Chunk, ...]}` with text joined from chunks.parquet."""
    fp = REPO_ROOT / f"results/{arch_prefix}__test_1273/retrieval.jsonl"
    if not fp.exists():
        raise FileNotFoundError(f"missing {fp}")
    qid_set = set(sample_qids)
    out: dict[str, list[Chunk]] = {}
    for line in fp.read_text().splitlines():
        row = json.loads(line)
        if row["question_id"] not in qid_set:
            continue
        ids = row.get("retrieved_chunk_ids", [])
        scores = row.get("retrieved_chunk_scores", [])
        chunks: list[Chunk] = []
        for cid, sc in zip(ids, scores):
            if cid in chunks_df.index:
                r = chunks_df.loc[cid]
                chunks.append(Chunk(
                    chunk_id=cid,
                    book_name=str(r.book_name) if "book_name" in r.index else "unknown",
                    text=str(r.text),
                    score=float(sc) if sc is not None else 0.0,
                ))
        out[row["question_id"]] = chunks
    return out

def build_rows(arch_label: str, arch_prefix: str, sample_df: pd.DataFrame) -> list[dict]:
    """Materialise the row dicts that run_lime_passage_batch expects.

    `load_medqa_4opt()` already returns `options` as a dict and `question_id`
    as `medqa_NNNNN`, so no re-parsing needed.
    """
    chunk_map = load_retrieval_for_sample(arch_prefix, sample_df.question_id.tolist())
    rows: list[dict] = []
    for _, r in sample_df.iterrows():
        rows.append({
            "question_id": r.question_id,
            "question": r.question,
            "options": r.options,  # already a dict from loader
            "gold_letter": r.answer_idx,
            "chunks": chunk_map.get(r.question_id, []),
            "architecture": arch_label,
        })
    return rows

# LLM callable wired to the runner-default generator config
LLM = partial(groq_complete, model=DEFAULT_MODEL, temperature=0.0, max_tokens=700)
def llm_callable(prompt: str):
    return LLM(prompt)

print("\nReady. LLM callable wired to:", DEFAULT_MODEL, "· T=0 · max_tokens=700")

chunks.parquet : 67,599 chunks indexed

Ready. LLM callable wired to: llama-3.3-70b-versatile · T=0 · max_tokens=700


## 4. Stage A — Smoke (3 questions × Multi-Hop only)

Each Multi-Hop question has ≤ 15 chunks, so 3 × ~16 = ~48 Groq calls (1 full hits cache, 15 LOO fresh per Q). Should take ~2–4 minutes.

In [4]:
RUN_SMOKE = True
RUN_SMALL = False
RUN_FULL  = False

ARCH_TARGETS = {
    "exp_02_naive_rag": "exp_02_naive_rag",
    "exp_04_hybrid_rag": "exp_04_hybrid_rag",
    "exp_05_multi_hop_rag": "exp_05_multi_hop_rag",
}

OUT_DIR = REPO_ROOT / "results" / "exp_10_lime_passage"

if RUN_SMOKE:
    smoke_df = SAMPLE.head(3).copy()
    arch_label = arch_prefix = "exp_05_multi_hop_rag"
    rows = build_rows(arch_label, arch_prefix, smoke_df)
    print(f"Smoke surface: 3 questions × Multi-Hop")
    print(f"Chunks-per-question on these 3: {[len(r['chunks']) for r in rows]}")
    summary = run_lime_passage_batch(
        rows=rows,
        output_path=OUT_DIR / "smoke_3.jsonl",
        llm_callable=llm_callable,
    )
    print("\n=== Smoke summary ===")
    print(json.dumps(summary, indent=2))
else:
    print("RUN_SMOKE = False — skipping Stage A")

Smoke surface: 3 questions × Multi-Hop
Chunks-per-question on these 3: [7, 10, 13]


LIME-LOO:   0%|          | 0/3 [00:00<?, ?it/s]


=== Smoke summary ===
{
  "output_path": "/Users/rajak/Workstation/Projects/myGitHub/thesis-project/results/exp_10_lime_passage/smoke_3.jsonl",
  "n_rows_written_this_run": 3,
  "n_rows_already_done": 0,
  "wall_time_s_this_run": 17.47453498840332
}


**Acceptance gates (Stage A)**:
1. 3 rows written to `smoke_3.jsonl` without error.
2. Wall time < 10 min — confirms LOO scales reasonably for Multi-Hop's k=15.
3. At least one question shows `top_chunk_by_correctness != None` (i.e., LOO of some chunk actually flipped correctness; not a trivial-everything-zero result).
4. `full_pred_letter` for each row matches the original EXP_05 prediction (sanity cache check).

In [5]:
if RUN_SMOKE:
    rows = [json.loads(l) for l in (OUT_DIR / "smoke_3.jsonl").read_text().splitlines()]
    e5_preds = pd.DataFrame([
        json.loads(l) for l in (REPO_ROOT / "results/exp_05_multi_hop_rag__test_1273/predictions.jsonl").read_text().splitlines()
    ]).set_index("question_id")

    print(f"Smoke rows: {len(rows)}\n")
    for r in rows:
        qid = r["question_id"]
        cached_full = e5_preds.loc[qid, "pred_letter"] if qid in e5_preds.index else "(missing)"
        agree = r["full_pred_letter"] == cached_full
        print(f"  {qid:>12s}  gold={r['gold_letter']}  full_pred={r['full_pred_letter']}  cached_E5={cached_full}  agree={'\u2713' if agree else '\u2717'}")
        print(f"      n_passages={r['n_passages']}  full_correct={r['full_correct']}")
        # Per-passage attributions
        for p in r["passages"]:
            marker = " " if p["correctness_attribution"] == 0 else ("+" if p["correctness_attribution"] > 0 else "-")
            print(f"      rank={p['rank']:>2d} chunk={p['chunk_id'][:40]:<40} corr={marker}{abs(p['correctness_attribution'])}  same_letter={p['sameletter_attribution']}")
        print(f"      top_chunk_by_correctness: {r['top_chunk_by_correctness']}")
        print(f"      top_chunk_by_sameletter : {r['top_chunk_by_sameletter']}")
        print()

Smoke rows: 3

   medqa_11615  gold=B  full_pred=B  cached_E5=B  agree=✓
      n_passages=7  full_correct=True
      rank= 0 chunk=InternalMed_Harrison_chunk_00133         corr= 0  same_letter=0
      rank= 1 chunk=InternalMed_Harrison_chunk_02964         corr= 0  same_letter=0
      rank= 2 chunk=InternalMed_Harrison_chunk_02966         corr= 0  same_letter=0
      rank= 3 chunk=InternalMed_Harrison_chunk_02967         corr= 0  same_letter=0
      rank= 4 chunk=First_Aid_Step2_chunk_00143              corr= 0  same_letter=0
      rank= 5 chunk=First_Aid_Step1_chunk_00170              corr= 0  same_letter=0
      rank= 6 chunk=Surgery_Schwartz_chunk_08520             corr= 0  same_letter=0
      top_chunk_by_correctness: None
      top_chunk_by_sameletter : None

   medqa_12517  gold=C  full_pred=C  cached_E5=C  agree=✓
      n_passages=10  full_correct=True
      rank= 0 chunk=Obstentrics_Williams_chunk_03088         corr= 0  same_letter=0
      rank= 1 chunk=Obstentrics_Williams_chun

## 4b. Stage A2 — Naive smoke (k=5 sanity diagnostic)

The Multi-Hop smoke (Stage A) returned **all-zero attribution** on all 3 questions × ~10 chunks. Two interpretations:

- **Structural**: LIME-LOO can't attribute distributed grounding at k=15. Removing 1 of 15 chunks rarely flips the answer because the other 14 carry redundant signal.
- **Multi-Hop-specific**: Multi-Hop's iterative retrieval produces high redundancy; smaller-k architectures (Naive, Hybrid) at k=5 should show attribution.

This cell runs the same 3 questions through **Naive (k=5)** to disambiguate. With k=5, each chunk carries ~20 % of the evidence — far more attribution pressure per chunk. If Naive shows non-zero attribution and Multi-Hop doesn't, the smoke result is a **Multi-Hop redundancy finding** worth keeping (thesis-publishable). If Naive is also all-zero, the issue is structural and EXP_11 / EXP_12 may need a methodology pivot.

Cost: 3 questions × 5 LOO ≈ 15 fresh Groq calls, < 1 min.

In [6]:
if RUN_SMOKE:
    smoke_df = SAMPLE.head(3).copy()
    arch_label = arch_prefix = "exp_02_naive_rag"
    rows = build_rows(arch_label, arch_prefix, smoke_df)
    print(f"Naive smoke: 3 questions × Naive (k=5)")
    print(f"Chunks-per-question on these 3: {[len(r['chunks']) for r in rows]}")
    summary = run_lime_passage_batch(
        rows=rows,
        output_path=OUT_DIR / "smoke_3_naive.jsonl",
        llm_callable=llm_callable,
    )
    print("\n=== Naive smoke summary ===")
    print(json.dumps(summary, indent=2))

    # Inspect Naive smoke
    naive_rows = [json.loads(l) for l in (OUT_DIR / "smoke_3_naive.jsonl").read_text().splitlines()]
    e2_preds = pd.DataFrame([
        json.loads(l) for l in (REPO_ROOT / "results/exp_02_naive_rag__test_1273/predictions.jsonl").read_text().splitlines()
    ]).set_index("question_id")

    print(f"\nNaive smoke rows: {len(naive_rows)}\n")
    for r in naive_rows:
        qid = r["question_id"]
        cached_full = e2_preds.loc[qid, "pred_letter"] if qid in e2_preds.index else "(missing)"
        agree = r["full_pred_letter"] == cached_full
        print(f"  {qid:>12s}  gold={r['gold_letter']}  full_pred={r['full_pred_letter']}  cached_E2={cached_full}  agree={'✓' if agree else '✗'}")
        print(f"      n_passages={r['n_passages']}  full_correct={r['full_correct']}")
        for p in r["passages"]:
            marker = " " if p["correctness_attribution"] == 0 else ("+" if p["correctness_attribution"] > 0 else "-")
            print(f"      rank={p['rank']:>2d} chunk={p['chunk_id'][:40]:<40} corr={marker}{abs(p['correctness_attribution'])}  same_letter={p['sameletter_attribution']}")
        print(f"      top_chunk_by_correctness: {r['top_chunk_by_correctness']}")
        print(f"      top_chunk_by_sameletter : {r['top_chunk_by_sameletter']}")
        print()

    # Diagnostic comparison
    mh_rows = [json.loads(l) for l in (OUT_DIR / "smoke_3.jsonl").read_text().splitlines()]
    def _any_attribution(r):
        return r["top_chunk_by_correctness"] is not None or r["top_chunk_by_sameletter"] is not None
    n_attr_mh = sum(1 for r in mh_rows if _any_attribution(r))
    n_attr_naive = sum(1 for r in naive_rows if _any_attribution(r))
    n_letter_changes_mh = sum(sum(1 for p in r["passages"] if p["sameletter_attribution"]) for r in mh_rows)
    n_letter_changes_naive = sum(sum(1 for p in r["passages"] if p["sameletter_attribution"]) for r in naive_rows)
    print(f"=== Diagnostic: any-attribution rate on smoke 3 questions ===")
    print(f"  Multi-Hop (k=15): {n_attr_mh}/3 questions with some attribution · {n_letter_changes_mh} total letter-changes across all LOO calls")
    print(f"  Naive    (k=5) : {n_attr_naive}/3 questions with some attribution · {n_letter_changes_naive} total letter-changes across all LOO calls")
    print()
    if n_letter_changes_mh == 0 and n_letter_changes_naive == 0:
        print("  → STRUCTURAL: both architectures all-zero on these 3 questions.")
        print("    The 3 smoke questions are likely all 'memorisation-only' or 'distractor-consensus' cases.")
        print("    Recommend: Stage B (50 Q × Multi-Hop) for a real estimate, OR pivot to subset-ablation methodology.")
    elif n_letter_changes_naive > n_letter_changes_mh:
        print("  → MULTI-HOP REDUNDANCY: Naive shows attribution where Multi-Hop doesn't.")
        print("    Publishable finding: Multi-Hop's k=15 chunks are too redundant for single-chunk LOO attribution.")
        print("    Recommend: scale up; reframe EXP_10 as 'redundancy vs grounding' analysis.")
    else:
        print("  → SMOKE WAS UNLUCKY: Multi-Hop shows attribution on at least one question.")
        print("    Recommend: scale up Stage B + Stage C with original methodology.")
else:
    print("RUN_SMOKE = False — skipping Stage A2")

Naive smoke: 3 questions × Naive (k=5)
Chunks-per-question on these 3: [5, 5, 5]


LIME-LOO:   0%|          | 0/3 [00:00<?, ?it/s]


=== Naive smoke summary ===
{
  "output_path": "/Users/rajak/Workstation/Projects/myGitHub/thesis-project/results/exp_10_lime_passage/smoke_3_naive.jsonl",
  "n_rows_written_this_run": 3,
  "n_rows_already_done": 0,
  "wall_time_s_this_run": 7.749388694763184
}

Naive smoke rows: 3

   medqa_11615  gold=B  full_pred=B  cached_E2=B  agree=✓
      n_passages=5  full_correct=True
      rank= 0 chunk=InternalMed_Harrison_chunk_00133         corr= 0  same_letter=0
      rank= 1 chunk=InternalMed_Harrison_chunk_02964         corr= 0  same_letter=0
      rank= 2 chunk=InternalMed_Harrison_chunk_02966         corr= 0  same_letter=0
      rank= 3 chunk=InternalMed_Harrison_chunk_02967         corr= 0  same_letter=0
      rank= 4 chunk=First_Aid_Step2_chunk_00143              corr= 0  same_letter=0
      top_chunk_by_correctness: None
      top_chunk_by_sameletter : None

   medqa_12517  gold=C  full_pred=C  cached_E2=C  agree=✓
      n_passages=5  full_correct=True
      rank= 0 chunk=Obstentr

## 4c. Stage A3 — Subset-sampling LIME smoke (methodology pivot)

LOO produced all-zero attribution on both Multi-Hop and Naive — the **STRUCTURAL** verdict from Stage A2. Pivoting to proper LIME with N=16 random subsets per question + ridge regression. Same 3 questions, Multi-Hop architecture.

**What's different**:
- LOO removed exactly 1 chunk per perturbation → all-zero attribution when chunks carry distributed signal.
- Subset-sampling removes a *random subset* of chunks per perturbation (each chunk in/out with p=0.5). Catches distributed grounding: a chunk that consistently appears in subsets where the LLM picks the full prediction gets a positive coefficient even if no single LOO flips the answer.
- Ridge regression (alpha=0.1) fits `score ≈ Σ w_i · mask_i + b`. The `w_i` are per-passage attributions.

**Cost**: 3 questions × 16 samples = 48 Groq calls. ~30 sec.

**Acceptance**:
1. `correctness_score_variance > 0` or `sameletter_score_variance > 0` on at least one question (i.e. some subset perturbation flips the answer → LIME has signal to attribute).
2. If variance is zero everywhere, even subset sampling can't find attribution → these 3 questions are pathologically robust to chunk perturbation, and we'd need a different sample to validate the method.

In [10]:
import importlib
import src.xai.lime_passage as _lp
importlib.reload(_lp)  # picks up new functions added after kernel start
from src.xai.lime_passage import run_subset_lime_batch

SUBSET_N = 16

if RUN_SMOKE:
    smoke_df = SAMPLE.head(3).copy()
    arch_label = arch_prefix = "exp_05_multi_hop_rag"
    rows = build_rows(arch_label, arch_prefix, smoke_df)
    print(f"Subset-LIME smoke: 3 questions × Multi-Hop, N={SUBSET_N} samples each")
    print(f"Chunks-per-question: {[len(r['chunks']) for r in rows]}")
    summary = run_subset_lime_batch(
        rows=rows,
        output_path=OUT_DIR / "smoke_3_subset.jsonl",
        llm_callable=llm_callable,
        n_samples=SUBSET_N,
        seed=42,
        alpha=0.1,
    )
    print("\n=== Subset-LIME smoke summary ===")
    print(json.dumps(summary, indent=2))

    # Inspect
    sub_rows = [json.loads(l) for l in (OUT_DIR / "smoke_3_subset.jsonl").read_text().splitlines()]
    print(f"\nSubset-LIME rows: {len(sub_rows)}\n")
    any_signal = False
    for r in sub_rows:
        print(f"  {r['question_id']:>12s}  gold={r['gold_letter']}  full_pred={r['full_pred_letter']}  full_correct={r['full_correct']}")
        print(f"      n_passages={r['n_passages']}  n_samples={r['n_samples']}")
        print(f"      score variance — correctness={r['correctness_score_variance']:.4f}  sameletter={r['sameletter_score_variance']:.4f}")
        # Letter histogram across samples
        from collections import Counter
        letter_hist = Counter(s['pred_letter'] for s in r['samples'])
        print(f"      letter histogram across {len(r['samples'])} samples: {dict(letter_hist)}")
        # Per-passage attribution (sort by abs(correctness_coef))
        passages = sorted(r['passages'], key=lambda p: -abs(p['correctness_coef']))
        print(f"      top 5 passages by |correctness_coef|:")
        for p in passages[:5]:
            print(f"        rank={p['rank']:>2d} chunk={p['chunk_id'][:38]:<38} corr_coef={p['correctness_coef']:+.4f}  same_coef={p['sameletter_coef']:+.4f}")
        print(f"      top_chunk_by_correctness: {r['top_chunk_by_correctness']}")
        print(f"      top_chunk_by_sameletter : {r['top_chunk_by_sameletter']}")
        if r['correctness_score_variance'] > 0 or r['sameletter_score_variance'] > 0:
            any_signal = True
        print()

    # Auto-verdict
    print("=== Verdict ===")
    if any_signal:
        print("  ✓ SIGNAL FOUND: at least one question has non-zero score variance.")
        print("    Subset-sampling LIME catches what LOO missed.")
        print("    Recommend: scale to Stage B (50 Q × Multi-Hop) and Stage C (200 Q × multi-arch).")
    else:
        print("  ✗ NO SIGNAL on these 3 questions even with subset sampling.")
        print("    The questions are pathologically robust to chunk perturbation.")
        print("    Recommend: try 3 different questions (e.g. ones where MultiHop_pred ≠ NoRAG_pred)")
        print("    before pivoting Phase 6 methodology.")
else:
    print("RUN_SMOKE = False — skipping Stage A3")

Subset-LIME smoke: 3 questions × Multi-Hop, N=16 samples each
Chunks-per-question: [7, 10, 13]


LIME-subset:   0%|          | 0/3 [00:00<?, ?it/s]


=== Subset-LIME smoke summary ===
{
  "output_path": "/Users/rajak/Workstation/Projects/myGitHub/thesis-project/results/exp_10_lime_passage/smoke_3_subset.jsonl",
  "method": "subset_lime",
  "n_samples_per_question": 16,
  "alpha": 0.1,
  "n_rows_written_this_run": 3,
  "n_rows_already_done": 0,
  "wall_time_s_this_run": 19.57962393760681
}

Subset-LIME rows: 3

   medqa_11615  gold=B  full_pred=B  full_correct=True
      n_passages=7  n_samples=16
      score variance — correctness=0.0000  sameletter=0.0000
      letter histogram across 16 samples: {'B': 16}
      top 5 passages by |correctness_coef|:
        rank= 0 chunk=InternalMed_Harrison_chunk_00133       corr_coef=+0.0000  same_coef=+0.0000
        rank= 1 chunk=InternalMed_Harrison_chunk_02964       corr_coef=+0.0000  same_coef=+0.0000
        rank= 2 chunk=InternalMed_Harrison_chunk_02966       corr_coef=+0.0000  same_coef=+0.0000
        rank= 3 chunk=InternalMed_Harrison_chunk_02967       corr_coef=+0.0000  same_coef=+0.0

## 4d. Stage A4 — Targeted "retrieval-fixed" diagnostic

Both Stage A2 (LOO) and Stage A3 (subset) showed zero signal on the first 3 sample questions. But all 3 are either **memorisation-only** (No-RAG also right) or **retrieval-distractor** (No-RAG right, RAG wrong) — *exactly* the question types where chunks can't carry attribution.

The Phase 4 regression analysis identified **101 / 1273 "retrieval-fixed" questions** on test_1273 — questions where No-RAG was wrong but Multi-Hop got the right answer. **These are the only questions where retrieval demonstrably did meaningful work** — and if LIME has any signal anywhere, it should be here.

This cell:
1. Filters SAMPLE to questions where No-RAG_pred ≠ gold AND Multi-Hop_pred = gold.
2. Picks the first 3 such questions.
3. Runs subset-LIME on them (N=16, Multi-Hop).
4. Reports the auto-verdict.

If signal shows up on this targeted subset, **LIME works on the right kind of question** — and we restrict Stage B/C to retrieval-fixed questions. If signal is still all-zero, chunk-level attribution is structurally dead for this benchmark and Phase 6 needs a methodology pivot to question-level analysis.

Cost: 3 × 16 = 48 Groq calls, ~30 sec.

In [11]:
if RUN_SMOKE:
    # Identify the "retrieval-fixed" subset: No-RAG_pred != gold AND MultiHop_pred = gold
    e1_preds = pd.DataFrame([
        json.loads(l) for l in (REPO_ROOT / "results/exp_01_base_llm__test_1273/predictions.jsonl").read_text().splitlines()
    ]).set_index("question_id")
    e5_preds = pd.DataFrame([
        json.loads(l) for l in (REPO_ROOT / "results/exp_05_multi_hop_rag__test_1273/predictions.jsonl").read_text().splitlines()
    ]).set_index("question_id")

    sample_aug = SAMPLE.copy()
    sample_aug["norag_correct"] = sample_aug.question_id.map(e1_preds.is_correct)
    sample_aug["mhop_correct"] = sample_aug.question_id.map(e5_preds.is_correct)
    retrieval_fixed = sample_aug[(sample_aug.norag_correct == False) & (sample_aug.mhop_correct == True)]
    print(f"Retrieval-fixed in SAMPLE (N=201): {len(retrieval_fixed)} questions")
    print(f"  (No-RAG wrong AND Multi-Hop right — the questions where chunks demonstrably did work)")
    if len(retrieval_fixed) < 3:
        print(f"  ⚠ fewer than 3 retrieval-fixed questions in sample; widening to full test_1273")
        # Fallback: pull retrieval-fixed qids from the full test set
        full_test_qids = set(e1_preds.index)
        rf_full = e1_preds[(e1_preds.is_correct == False)].index.intersection(
            e5_preds[(e5_preds.is_correct == True)].index
        )
        print(f"  retrieval-fixed in full test_1273: {len(rf_full)}")
        smoke_target_df = medqa[medqa.question_id.isin(list(rf_full)[:3])]
    else:
        smoke_target_df = retrieval_fixed.head(3)

    print(f"\nTarget smoke set: {len(smoke_target_df)} questions")
    print(smoke_target_df[["question_id", "question"]].assign(
        question=smoke_target_df.question.str[:80] + "…"
    ).to_string(index=False))

    arch_label = arch_prefix = "exp_05_multi_hop_rag"
    rows = build_rows(arch_label, arch_prefix, smoke_target_df)
    print(f"\nChunks-per-question: {[len(r['chunks']) for r in rows]}")
    summary = run_subset_lime_batch(
        rows=rows,
        output_path=OUT_DIR / "smoke_3_retrievalfixed_subset.jsonl",
        llm_callable=llm_callable,
        n_samples=SUBSET_N,
        seed=42,
        alpha=0.1,
    )
    print("\n=== Targeted (retrieval-fixed) subset-LIME summary ===")
    print(json.dumps(summary, indent=2))

    rf_rows = [json.loads(l) for l in (OUT_DIR / "smoke_3_retrievalfixed_subset.jsonl").read_text().splitlines()]
    print(f"\nTarget rows: {len(rf_rows)}\n")
    any_signal = False
    from collections import Counter
    for r in rf_rows:
        print(f"  {r['question_id']:>12s}  gold={r['gold_letter']}  full_pred={r['full_pred_letter']}  full_correct={r['full_correct']}")
        print(f"      n_passages={r['n_passages']}  n_samples={r['n_samples']}")
        print(f"      variance — corr={r['correctness_score_variance']:.4f}  same={r['sameletter_score_variance']:.4f}")
        hist = Counter(s['pred_letter'] for s in r['samples'])
        print(f"      letter histogram: {dict(hist)}")
        if r['correctness_score_variance'] > 0 or r['sameletter_score_variance'] > 0:
            any_signal = True
            passages = sorted(r['passages'], key=lambda p: -abs(p['correctness_coef']))
            print(f"      top 5 passages by |correctness_coef|:")
            for p in passages[:5]:
                print(f"        rank={p['rank']:>2d} chunk={p['chunk_id'][:38]:<38} corr={p['correctness_coef']:+.4f}  same={p['sameletter_coef']:+.4f}")
            print(f"      top_chunk_by_correctness: {r['top_chunk_by_correctness']}")
            print(f"      top_chunk_by_sameletter : {r['top_chunk_by_sameletter']}")
        print()

    print("=== Stage A4 verdict ===")
    if any_signal:
        print("  ✓ SIGNAL FOUND on retrieval-fixed questions.")
        print("    Recommendation: scale Stage B/C to retrieval-fixed subset only (~101 questions on test_1273).")
        print("    EXP_10 will report chunk-level attribution restricted to questions where retrieval mattered.")
        print("    Methodology footnote: 'LIME signal is only well-defined on questions where retrieval changed")
        print("    the answer; on memorisation-only and retrieval-distractor cases, the LLM's prediction is")
        print("    robust to per-chunk perturbation.'")
    else:
        print("  ✗ NO SIGNAL even on retrieval-fixed questions.")
        print("    Chunk-level LIME attribution is structurally dead on this benchmark.")
        print("    Recommendation: pivot Phase 6 to question-level binary 'did retrieval matter' analysis;")
        print("    drop chunk-level LIME/SHAP/agreement; replace EXP_12 with a different confidence signal.")
else:
    print("RUN_SMOKE = False — skipping Stage A4")

Retrieval-fixed in SAMPLE (N=201): 15 questions
  (No-RAG wrong AND Multi-Hop right — the questions where chunks demonstrably did work)

Target smoke set: 3 questions
question_id                                                                          question
medqa_12276 A 22-year-old woman presents to the emergency department with a headache. She ha…
medqa_12222 A 25-year-old man presents to his primary care provider complaining of several w…
medqa_11672 A 60-year-old woman presents to a physician for worsening shortness of breath an…

Chunks-per-question: [11, 13, 10]


LIME-subset:   0%|          | 0/3 [00:00<?, ?it/s]


=== Targeted (retrieval-fixed) subset-LIME summary ===
{
  "output_path": "/Users/rajak/Workstation/Projects/myGitHub/thesis-project/results/exp_10_lime_passage/smoke_3_retrievalfixed_subset.jsonl",
  "method": "subset_lime",
  "n_samples_per_question": 16,
  "alpha": 0.1,
  "n_rows_written_this_run": 3,
  "n_rows_already_done": 0,
  "wall_time_s_this_run": 22.90513586997986
}

Target rows: 3

   medqa_12276  gold=B  full_pred=B  full_correct=True
      n_passages=11  n_samples=16
      variance — corr=0.0586  same=0.0586
      letter histogram: {'B': 15, 'C': 1}
      top 5 passages by |correctness_coef|:
        rank= 4 chunk=InternalMed_Harrison_chunk_15421       corr=-0.3131  same=-0.3131
        rank= 7 chunk=Obstentrics_Williams_chunk_04948       corr=+0.3060  same=+0.3060
        rank= 2 chunk=First_Aid_Step2_chunk_00378            corr=-0.3051  same=-0.3051
        rank= 3 chunk=Neurology_Adams_chunk_00716            corr=-0.2306  same=-0.2306
        rank= 9 chunk=Neurology_A

## 5. Stage B — Subset-LIME on retrieval-changed × Multi-Hop (canonical EXP_10 run)

Stage A4 confirmed: subset-sampling LIME shows signal on questions where retrieval changed the LLM's answer (No-RAG_pred ≠ Multi-Hop_pred), but not on memorisation or distractor-consensus cases. Stage B runs the canonical EXP_10 pass on the full **retrieval-changed** subset for Multi-Hop on test_1273.

**Sample definition**: `e1_preds[qid].pred_letter ≠ e5_preds[qid].pred_letter` for all qids in the test split. This is ~174 questions (101 retrieval-fixed + 73 retrieval-broken; see [`docs/output_notes/04e_exp05_output.md` §3.5](../docs/output_notes/04e_exp05_output.md)).

**Including both fixes and breaks**:
- **Fixes** (No-RAG wrong, Multi-Hop right): positive `correctness_coef` identifies *supporting* chunks.
- **Breaks** (No-RAG right, Multi-Hop wrong): negative `correctness_coef` identifies *distractor* chunks (when they're present, the LLM is wrong).
- Both rows together let EXP_12 LIME-SHAP agreement compute on the full retrieval-mattered subset; Phase 7 confidence-aware rejection uses the magnitude distribution as a signal.

**Cost**: ~174 × 16 = 2,784 fresh Groq calls. ~22 min wall time at ~0.5 s/call. $0 (Groq free tier). Resumable.

Flip `RUN_STAGE_B = True` to run.

In [12]:
RUN_STAGE_B = True  # flip after Stage A4 verdict is SIGNAL FOUND

if RUN_STAGE_B:
    # Identify retrieval-changed questions on test_1273: NoRAG_pred != MultiHop_pred
    e1 = pd.DataFrame([
        json.loads(l) for l in (REPO_ROOT / "results/exp_01_base_llm__test_1273/predictions.jsonl").read_text().splitlines()
    ]).set_index("question_id")[["pred_letter", "is_correct"]].rename(columns={"pred_letter":"e1_pred","is_correct":"e1_correct"})
    e5 = pd.DataFrame([
        json.loads(l) for l in (REPO_ROOT / "results/exp_05_multi_hop_rag__test_1273/predictions.jsonl").read_text().splitlines()
    ]).set_index("question_id")[["pred_letter", "is_correct"]].rename(columns={"pred_letter":"e5_pred","is_correct":"e5_correct"})
    pair = e1.join(e5)
    retrieval_changed_qids = pair[pair.e1_pred != pair.e5_pred].index.tolist()

    n_fixes = ((pair.e1_correct == False) & (pair.e5_correct == True)).sum()
    n_breaks = ((pair.e1_correct == True) & (pair.e5_correct == False)).sum()
    n_other = len(retrieval_changed_qids) - n_fixes - n_breaks  # both wrong but differ
    print(f"Retrieval-changed (NoRAG_pred ≠ Multi-Hop_pred) on test_1273:")
    print(f"  total          : {len(retrieval_changed_qids)}")
    print(f"  fixes  (NR✗ MH✓): {n_fixes}")
    print(f"  breaks (NR✓ MH✗): {n_breaks}")
    print(f"  other  (both ✗ different letters): {n_other}")

    # Build target df from medqa (full set, not the 200-sample)
    target_df = medqa[medqa.question_id.isin(retrieval_changed_qids)].copy()
    print(f"\nBuilding rows for {len(target_df)} questions...")
    arch_label = arch_prefix = "exp_05_multi_hop_rag"
    rows = build_rows(arch_label, arch_prefix, target_df)
    # Forecast
    total_chunks = sum(len(r["chunks"]) for r in rows)
    print(f"  total chunks across questions: {total_chunks}")
    print(f"  Groq calls to make: {len(rows)} × {SUBSET_N} = {len(rows) * SUBSET_N}")

    summary = run_subset_lime_batch(
        rows=rows,
        output_path=OUT_DIR / "stage_b_retrievalchanged_mhop.jsonl",
        llm_callable=llm_callable,
        n_samples=SUBSET_N,
        seed=42,
        alpha=0.1,
    )
    print("\n=== Stage B summary ===")
    print(json.dumps(summary, indent=2))
else:
    print("RUN_STAGE_B = False — flip to True after Stage A4 = SIGNAL FOUND")

Retrieval-changed (NoRAG_pred ≠ Multi-Hop_pred) on test_1273:
  total          : 205
  fixes  (NR✗ MH✓): 101
  breaks (NR✓ MH✗): 73
  other  (both ✗ different letters): 31

Building rows for 205 questions...
  total chunks across questions: 2410
  Groq calls to make: 205 × 16 = 3280


LIME-subset:   0%|          | 0/205 [00:00<?, ?it/s]


=== Stage B summary ===
{
  "output_path": "/Users/rajak/Workstation/Projects/myGitHub/thesis-project/results/exp_10_lime_passage/stage_b_retrievalchanged_mhop.jsonl",
  "method": "subset_lime",
  "n_samples_per_question": 16,
  "alpha": 0.1,
  "n_rows_written_this_run": 205,
  "n_rows_already_done": 0,
  "wall_time_s_this_run": 1448.650512933731
}


## ~~6. Stage C — Full (200 questions × N architectures)~~ **OBSOLETE — DO NOT RUN**

This stage predates the methodology pivot to subset-LIME. It uses:
1. The **LOO method** — Stage A2 confirmed this produces zero attribution (STRUCTURAL verdict).
2. **Random 200-question SAMPLE** — most are memorisation cases where chunks don't change the answer; LIME has nothing to attribute.

The canonical EXP_10 deliverable is **Stage B** (subset-LIME on the 205 retrieval-changed Multi-Hop questions, 78.5 % signal density). Stage C as designed would run for ~25 min and produce all zeros. **Skip it.**

If we later want to extend EXP_10 to Naive / Hybrid (k=5 architectures), we'd add a "Stage C2" that uses subset-LIME on each architecture's retrieval-changed subset (~149 Naive, ~154 Hybrid, ~104 Sparse questions, k=5). For now, Multi-Hop alone is the thesis-priority target.

In [ ]:
# ========================================================================
# Stage C is OBSOLETE — see the markdown cell above.
# Setting RUN_FULL = False ensures this never executes by accident.
# Canonical EXP_10 output: results/exp_10_lime_passage/stage_b_retrievalchanged_mhop.jsonl
# ========================================================================
RUN_FULL = False
print("Stage C skipped — Stage B is the canonical EXP_10 deliverable.")
print("To extend EXP_10 to Naive/Hybrid later, add a new Stage C2 with subset-LIME on each")
print("architecture's retrieval-changed subset (do NOT re-enable the legacy LOO code below).")

# Legacy code preserved for audit trail; never executed:
if False:
    ARCHITECTURES_TO_RUN = ["exp_05_multi_hop_rag"]
    for arch in ARCHITECTURES_TO_RUN:
        print(f"\n--- {arch} ---")
        rows = build_rows(arch, arch, SAMPLE)
        summary = run_lime_passage_batch(  # ← old LOO method
            rows=rows,
            output_path=OUT_DIR / f"full_200_{arch}.jsonl",
            llm_callable=llm_callable,
        )
        print(json.dumps(summary, indent=2))

## 7. Inspect — aggregate distributions per architecture

In [ ]:
# Find all completed JSONL files and aggregate
for fp in sorted(OUT_DIR.glob("*.jsonl")) if OUT_DIR.exists() else []:
    rows = [json.loads(l) for l in fp.read_text().splitlines()]
    if not rows:
        continue
    print(f"\n=== {fp.name}  (n={len(rows)}) ===")
    df = pd.DataFrame([
        {
            "question_id": r["question_id"],
            "architecture": r["architecture"],
            "full_correct": r["full_correct"],
            "n_passages": r["n_passages"],
            "has_top_correct": r["top_chunk_by_correctness"] is not None,
            "has_top_sameletter": r["top_chunk_by_sameletter"] is not None,
            "n_helpful": sum(1 for p in r["passages"] if p["correctness_attribution"] > 0),
            "n_hurtful": sum(1 for p in r["passages"] if p["correctness_attribution"] < 0),
            "n_letter_changes": sum(1 for p in r["passages"] if p["sameletter_attribution"] == 1),
        }
        for r in rows
    ])
    print(f"  full_correct rate           : {df.full_correct.mean():.3f}")
    print(f"  rows with a top-correct chunk: {df.has_top_correct.sum()}/{len(df)} = {df.has_top_correct.mean():.3f}")
    print(f"  rows with a top-sameletter   : {df.has_top_sameletter.sum()}/{len(df)} = {df.has_top_sameletter.mean():.3f}")
    print(f"  mean #helpful chunks per Q   : {df.n_helpful.mean():.2f}")
    print(f"  mean #hurtful chunks per Q   : {df.n_hurtful.mean():.2f}")
    print(f"  mean #letter-changes per Q   : {df.n_letter_changes.mean():.2f}")
    print(f"  mean n_passages per Q        : {df.n_passages.mean():.2f}")

---

**Next.** EXP_11 (passage-level SHAP). After EXP_10 + EXP_11 land on the same sample of 200 questions, EXP_12 computes the per-question top-1 / top-3 overlap between LIME and SHAP rankings and correlates it with Faithfulness / accuracy.